# Module 46: Quantization Recipes

Explore PyTorch quantization techniques to reduce model size and speed up inference.

**Topics:**
- Dynamic quantization (no calibration needed)
- Static quantization (calibrate activation ranges)
- Model size comparison

In [ ]:
import torch
import torch.nn as nn
import torch.ao.quantization as quant

# Simple model for quantization
model = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
model.eval()
print(f"FP32 parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Dynamic quantization: quantize Linear layers to INT8
model_dynamic = quant.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

# Compare sizes
import io
def model_size_kb(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.tell() / 1024

fp32_kb = model_size_kb(model)
int8_kb = model_size_kb(model_dynamic)
print(f"FP32: {fp32_kb:.1f} KB")
print(f"INT8: {int8_kb:.1f} KB")
print(f"Reduction: {(1 - int8_kb / fp32_kb) * 100:.1f}%")

In [ ]:
# Verify output agreement
x = torch.randn(16, 512)
with torch.no_grad():
    fp32_out = model(x)
    int8_out = model_dynamic(x)

agreement = (fp32_out.argmax(1) == int8_out.argmax(1)).float().mean()
print(f"Prediction agreement: {agreement:.1%}")
print(f"Max absolute diff: {(fp32_out - int8_out).abs().max():.6f}")

In [ ]:
# Benchmark inference speed
import time

def bench(m, x, runs=200):
    m.eval()
    for _ in range(20):  # warm-up
        with torch.no_grad(): m(x)
    t0 = time.perf_counter()
    for _ in range(runs):
        with torch.no_grad(): m(x)
    return (time.perf_counter() - t0) / runs * 1000

x = torch.randn(1, 512)
fp32_ms = bench(model, x)
int8_ms = bench(model_dynamic, x)
print(f"FP32: {fp32_ms:.3f} ms")
print(f"INT8: {int8_ms:.3f} ms")
print(f"Speedup: {fp32_ms / int8_ms:.2f}x")

## Next Steps

- Try static quantization with `quant.prepare` + calibration for better speed
- Explore QAT for accuracy-sensitive models
- See `dynamic_quantization.py` and `static_quantization.py` for full examples
- Check out [torchao](https://github.com/pytorch/ao) for INT4 and FP8 quantization